In [57]:
from google.colab import auth
auth.authenticate_user()

from google.cloud import bigquery
client = bigquery.Client(project='tough-country-504306-p8')
print("Connected!")

Connected!


In [44]:
query = """
SELECT
  complaint_id,
  product,
  issue,
  company_name,
  state,
  date_received,
  consumer_complaint_narrative
FROM
  `bigquery-public-data.cfpb_complaints.complaint_database`
WHERE
  consumer_complaint_narrative IS NOT NULL
LIMIT 50000
"""

df = client.query(query).to_dataframe()
print(df.shape)
df.head()

(50000, 7)


,complaint_id,product,issue,company_name,state,date_received,consumer_complaint_narrative
0,2263014,Debt collection,Cont'd attempts collect debt not owed,ERC,AL,2016-12-23,I paid charter communications through Enhanced...
1,3175934,Debt collection,Attempts to collect debt not owed,ERC,MD,2019-03-11,The company accepted a reduced settlement over...
2,2204999,Debt collection,Cont'd attempts collect debt not owed,ERC,PA,2016-11-11,I have a major complain concerning lack of fle...
3,1486242,Debt collection,Cont'd attempts collect debt not owed,ERC,IL,2015-07-23,I have paid for the same debt for XXXX in XXXX...
4,2752883,Debt collection,Attempts to collect debt not owed,ERC,WY,2017-12-12,I received a collection statement from ERC for...


In [45]:
category_map = {
    'Credit reporting': 'Credit reporting, credit repair services, or other personal consumer reports',
    'Credit card': 'Credit card or prepaid card',
    'Prepaid card': 'Credit card or prepaid card',
    'Payday loan': 'Payday loan, title loan, or personal loan',
    'Consumer Loan': 'Payday loan, title loan, or personal loan',
    'Money transfers': 'Money transfer, virtual currency, or money service',
    'Bank account or service': 'Checking or savings account',
}

df['product_clean'] = df['product'].replace(category_map)

# Drop categories that are still too rare to model meaningfully
counts = df['product_clean'].value_counts()
valid_categories = counts[counts >= 100].index
df_clean = df[df['product_clean'].isin(valid_categories)].copy()

print(df_clean['product_clean'].value_counts())
print("\nRows kept:", len(df_clean), "out of", len(df))

product_clean
Credit reporting, credit repair services, or other personal consumer reports    26199
Debt collection                                                                  6120
Credit card or prepaid card                                                      5508
Checking or savings account                                                      3726
Mortgage                                                                         3197
Money transfer, virtual currency, or money service                               1831
Student loan                                                                     1663
Payday loan, title loan, or personal loan                                        1086
Vehicle loan or lease                                                             663
Name: count, dtype: int64

Rows kept: 49993 out of 50000


In [46]:
from sklearn.model_selection import train_test_split

X = df['consumer_complaint_narrative']
y = df['product']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train size:", X_train.shape[0])
print("Test size:", X_test.shape[0])
print("\nClass distribution:\n", y.value_counts())

Train size: 40000
Test size: 10000

Class distribution:
 product
Credit reporting, credit repair services, or other personal consumer reports    24831
Debt collection                                                                  6120
Credit card or prepaid card                                                      4379
Mortgage                                                                         3197
Checking or savings account                                                      3013
Money transfer, virtual currency, or money service                               1784
Student loan                                                                     1663
Credit reporting                                                                 1368
Credit card                                                                      1089
Bank account or service                                                           713
Payday loan, title loan, or personal loan                                  

In [47]:
X = df_clean['consumer_complaint_narrative']
y = df_clean['product_clean']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train size:", X_train.shape[0])
print("Test size:", X_test.shape[0])

Train size: 39994
Test size: 9999


In [48]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

# Vectorize the text
vectorizer = TfidfVectorizer(max_features=10000, stop_words='english', ngram_range=(1,2))
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

# Train the classifier
clf = LogisticRegression(max_iter=1000, class_weight='balanced')
clf.fit(X_train_tfidf, y_train)

# Predict and evaluate
y_pred = clf.predict(X_test_tfidf)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

Accuracy: 0.8183818381838184

Classification Report:

                                                                              precision    recall  f1-score   support

                                                 Checking or savings account       0.78      0.80      0.79       745
                                                 Credit card or prepaid card       0.74      0.79      0.77      1102
Credit reporting, credit repair services, or other personal consumer reports       0.95      0.84      0.89      5240
                                                             Debt collection       0.69      0.78      0.73      1224
                          Money transfer, virtual currency, or money service       0.66      0.78      0.72       366
                                                                    Mortgage       0.84      0.91      0.87       639
                                   Payday loan, title loan, or personal loan       0.37      0.54      0.44       217
 

In [49]:
import joblib

joblib.dump(vectorizer, "tfidf_vectorizer.pkl")
joblib.dump(clf, "logreg_model.pkl")

print("Saved!")

Saved!


In [50]:
from google.colab import files
files.download("tfidf_vectorizer.pkl")
files.download("logreg_model.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [51]:
!pip install vaderSentiment -q

from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

analyzer = SentimentIntensityAnalyzer()

df_clean['sentiment_score'] = df_clean['consumer_complaint_narrative'].apply(
    lambda text: analyzer.polarity_scores(text)['compound']
)

def classify_sentiment(score):
    if score >= 0.05:
        return 'positive'
    elif score <= -0.05:
        return 'negative'
    else:
        return 'neutral'

df_clean['sentiment_label'] = df_clean['sentiment_score'].apply(classify_sentiment)

print(df_clean['sentiment_label'].value_counts())
print("\nAverage sentiment by product:\n")
print(df_clean.groupby('product_clean')['sentiment_score'].mean().sort_values())

sentiment_label
negative    23982
positive    23766
neutral      2245
Name: count, dtype: int64

Average sentiment by product:

product_clean
Debt collection                                                                -0.247841
Checking or savings account                                                    -0.187622
Money transfer, virtual currency, or money service                             -0.135412
Mortgage                                                                       -0.015556
Vehicle loan or lease                                                          -0.013152
Credit reporting, credit repair services, or other personal consumer reports    0.030098
Payday loan, title loan, or personal loan                                       0.051258
Credit card or prepaid card                                                     0.058504
Student loan                                                                    0.097660
Name: sentiment_score, dtype: float64


In [52]:
export_df = df_clean[['complaint_id', 'product_clean', 'issue', 'company_name',
                        'state', 'date_received', 'sentiment_score', 'sentiment_label']].copy()
export_df['predicted_correct_sample'] = None  # placeholder, optional

export_df.to_csv('cfpb_complaints_for_powerbi.csv', index=False)

from google.colab import files
files.download('cfpb_complaints_for_powerbi.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [53]:
import plotly.express as px

state_counts = df_clean.groupby('state').size().reset_index(name='complaint_count')

fig = px.choropleth(
    state_counts,
    locations='state',
    locationmode='USA-states',
    color='complaint_count',
    scope='usa',
    color_continuous_scale='Blues',
    title='CFPB Complaint Volume by State'
)
fig.show()

In [54]:
app_code = '''
import streamlit as st
import pandas as pd
import plotly.express as px

st.set_page_config(page_title="CFPB Complaint Intelligence", layout="wide")
st.title("Consumer Complaint Sentiment & Classification Pipeline")
st.markdown("Analysis of CFPB financial complaints: product classification, sentiment scoring, and geographic trends.")

df = pd.read_csv("cfpb_complaints_for_powerbi.csv")

col1, col2, col3 = st.columns(3)
col1.metric("Total Complaints Analyzed", f"{len(df):,}")
col2.metric("Classifier Accuracy", "81.8%")
col3.metric("Negative Sentiment Share", f"{(df['sentiment_label']=='negative').mean()*100:.1f}%")

st.subheader("Complaint Volume by State")
state_counts = df.groupby('state').size().reset_index(name='complaint_count')
fig_map = px.choropleth(
    state_counts, locations='state', locationmode='USA-states',
    color='complaint_count', scope='usa', color_continuous_scale='Blues'
)
st.plotly_chart(fig_map, use_container_width=True)

st.subheader("Sentiment by Product Category")
sentiment_by_product = df.groupby('product_clean')['sentiment_score'].mean().sort_values()
st.bar_chart(sentiment_by_product)
'''

with open('app.py', 'w') as f:
    f.write(app_code)

print("app.py created")

app.py created


In [55]:
files.download('app.py')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [56]:
import sklearn
print(sklearn.__version__)

1.6.1
